In [ ]:
%run ./1_config


In [ ]:

from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

class BronzeEventsLoader:
    def __init__(self):
        self.c = conf
        self.fqn = self.c.table_fqn(self.c.bronze_events)

    def load_csv(self):
        path = self.c.raw_path
        print(f"Reading {path}")
        df = (spark.read.option("header", True).csv(path)
              .withColumn("event_ts", F.to_timestamp("event_ts"))
              .withColumn("amount", F.col("amount").cast(DoubleType()))
              .withColumn("event_date", F.to_date("event_ts"))
              .withColumn("_ingested_at", F.current_timestamp()))
        cutoff = F.date_sub(F.current_date(), self.c.lookback_days)
        df = df.filter(F.col("event_date") >= cutoff)
        return df

    def run(self):
        df = self.load_csv()
        cnt = df.count()
        df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(self.fqn)
        print(f"✅ bronze_events appended {cnt} rows")

    def validate(self):
        n = spark.table(self.fqn).count()
        print(f"🔎 bronze_events total rows: {n}")

loader = BronzeEventsLoader()
loader.run()
loader.validate()

